In [111]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone

import os
import sys
sys.path.append(os.path.abspath('./src'))

from db_functions import DotaDB

db = DotaDB(local=True)

In [112]:
query = 'SELECT * FROM main_metadata'
matches = db.query_select_to_df(query, table='main_metadata')

radiant_rows = matches[['match_id', 'radiant_team_id', 'radiant_win', 'start_date_time']].copy()
radiant_rows = radiant_rows.rename(columns={'radiant_team_id': 'team_id', 'radiant_win': 'won'})
radiant_rows = radiant_rows.dropna(subset=['team_id'])

dire_rows = matches[['match_id', 'dire_team_id', 'radiant_win', 'start_date_time']].copy()
dire_rows = dire_rows.rename(columns={'dire_team_id': 'team_id', 'radiant_win': 'won'})
dire_rows = dire_rows.dropna(subset=['team_id'])
dire_rows['won'] = ~dire_rows['won']

team_history = (
    pd.concat([radiant_rows, dire_rows])
    .sort_values(['team_id', 'start_date_time'])
    .reset_index(drop=True)
)

# form = wins in last 5 games before this match (shift(1) excludes current match)
team_history['form'] = (
    team_history.groupby('team_id', group_keys=False)['won']
    .apply(lambda x: x.shift(1).rolling(window=10, min_periods=1).sum())
)

# merge radiant form
radiant_form = team_history[['match_id', 'team_id', 'form']].rename(
    columns={'team_id': 'radiant_team_id', 'form': 'radiant_form'}
)
matches = matches.merge(radiant_form, on=['match_id', 'radiant_team_id'], how='left')

# merge dire form
dire_form = team_history[['match_id', 'team_id', 'form']].rename(
    columns={'team_id': 'dire_team_id', 'form': 'dire_form'}
)
matches = matches.merge(dire_form, on=['match_id', 'dire_team_id'], how='left')

In [113]:
## Roster longevity
query = '''
    SELECT 
        pms.match_id,
        pms.hero_id,
        pms.is_radiant,
        pms.win,
        pms.account_id,
        mm.radiant_team_id,
        mm.dire_team_id,
        mm.start_date_time
    FROM player_match_stats pms
    INNER JOIN main_metadata mm ON mm.match_id = pms.match_id
'''
player_match_data = pd.DataFrame(
    db.query_select(query),
    columns=[
        'match_id',
        'hero_id',
        'is_radiant',
        'won',
        'account_id',
        'radiant_team_id',
        'dire_team_id',
        'start_date_time'
    ]
)

# build roster tuple per match per side — sorted for consistent comparison
roster_groups = (
    player_match_data
    .sort_values(['match_id', 'is_radiant', 'account_id'])
    .groupby(['match_id', 'is_radiant'])['account_id']
    .apply(tuple)
    .reset_index()
    .rename(columns={'account_id': 'roster_tuple'})
)

# merge radiant and dire rosters onto matches
radiant_rosters = roster_groups[roster_groups['is_radiant'] == True][['match_id', 'roster_tuple']].rename(
    columns={'roster_tuple': 'radiant_roster'}
)
dire_rosters = roster_groups[roster_groups['is_radiant'] == False][['match_id', 'roster_tuple']].rename(
    columns={'roster_tuple': 'dire_roster'}
)

matches = matches.merge(radiant_rosters, on='match_id', how='left')
matches = matches.merge(dire_rosters,    on='match_id', how='left')

radiant_team_rosters = matches[['match_id', 'radiant_team_id', 'radiant_roster', 'start_date_time']].rename(
    columns={'radiant_team_id': 'team_id', 'radiant_roster': 'roster'}
)
dire_team_rosters = matches[['match_id', 'dire_team_id', 'dire_roster', 'start_date_time']].rename(
    columns={'dire_team_id': 'team_id', 'dire_roster': 'roster'}
)

roster_history = (
    pd.concat([radiant_team_rosters, dire_team_rosters])
    .dropna(subset=['team_id'])
    .sort_values(['team_id', 'roster', 'start_date_time'])
    .reset_index(drop=True)
)

# cumcount gives games played with this exact roster before this match
# shift via cumcount is inherent — cumcount starts at 0 so first game = 0 experience
roster_history['roster_experience'] = roster_history.groupby(['team_id', 'roster']).cumcount() 

roster_first_game = (
    roster_history.groupby(['roster'])['start_date_time']
    .min()
    .reset_index()
    .rename(columns={'start_date_time': 'roster_first_game'})
)
roster_history = roster_history.merge(roster_first_game, on='roster', how='left')
roster_history['roster_age_days'] = (
    pd.to_datetime(roster_history['start_date_time']) -
    pd.to_datetime(roster_history['roster_first_game'])
).dt.days

# merge radiant roster experience
radiant_roster_exp = roster_history[['match_id', 'team_id', 'roster_experience', 'roster_age_days']].rename(
    columns={'team_id': 'radiant_team_id', 'roster_experience': 'radiant_roster_exp', 'roster_age_days': 'radiant_roster_age_days'}
)
matches = matches.merge(radiant_roster_exp, on=['match_id', 'radiant_team_id'], how='left')

# merge dire roster experience
dire_roster_exp = roster_history[['match_id', 'team_id', 'roster_experience', 'roster_age_days']].rename(
    columns={'team_id': 'dire_team_id', 'roster_experience': 'dire_roster_exp', 'roster_age_days': 'dire_roster_age_days'}
)
matches = matches.merge(dire_roster_exp, on=['match_id', 'dire_team_id'], how='left')

In [114]:
matches['average_rank'].describe()

count    169.000000
mean      74.556213
std        7.044804
min       52.000000
25%       73.000000
50%       75.000000
75%       81.000000
max       81.000000
Name: average_rank, dtype: float64

In [115]:
draft_strength_history = pd.read_csv('data/draft_strength.csv').rename(columns={'rad_draft_strength': 'radiant_draft_strength'})
matches = matches.merge(
    draft_strength_history[['match_id', 'radiant_draft_strength', 'dire_draft_strength']],
    on='match_id',
    how='left'    
)

In [116]:
rating_history = pd.read_csv('data/rating_history_30_25_100.csv')
avg_ratings = rating_history.groupby(['match_id', 'is_radiant'])['ordinal'].mean().unstack('is_radiant')
avg_ratings.columns = ['avg_dire_rating', 'avg_radiant_rating']
avg_ratings = avg_ratings.reset_index()
matches = matches.merge(avg_ratings, on='match_id', how='left')

In [117]:
features = [
    'radiant_draft_strength',
    'dire_draft_strength',
    'avg_radiant_rating',
    'avg_dire_rating',
    'radiant_roster_age_days',
    'dire_roster_age_days',
    'radiant_form',
    'dire_form',
    'radiant_roster_exp', 
    'dire_roster_exp', 
    'radiant_win'
]
test_matches = matches.dropna(
    subset=[
        'radiant_draft_strength',
        'dire_draft_strength',
        'avg_radiant_rating',
        'avg_dire_rating'
    ]
)
test_matches = test_matches[features]
test_matches = test_matches[test_matches['avg_radiant_rating'] != 0]
test_matches = test_matches[test_matches['avg_dire_rating'] != 0]
test_matches = test_matches.fillna(0)

In [118]:
test_matches['draft_diff'] = test_matches['radiant_draft_strength'] - test_matches['dire_draft_strength']
test_matches['rating_diff'] = test_matches['avg_radiant_rating'] - test_matches['avg_dire_rating']
test_matches['form_diff'] = test_matches['radiant_form'] - test_matches['dire_form']
test_matches['roster_exp_diff'] = np.log1p(test_matches['radiant_roster_exp']) - np.log1p(test_matches['dire_roster_exp'])
test_matches['roster_age_diff'] = np.log1p(test_matches['radiant_roster_age_days']) - np.log1p(test_matches['dire_roster_age_days'])
test_matches = test_matches.iloc[:, -6:]

In [119]:
test_matches

,radiant_win,draft_diff,rating_diff,form_diff,roster_exp_diff,roster_age_diff
2,False,0.000692,-3.028040,0.0,0.000000,0.000000
3,False,0.014958,3.010599,0.0,0.000000,0.000000
5,True,-0.015279,-0.930563,0.0,0.000000,0.000000
7,False,-0.027248,-3.020713,0.0,0.000000,0.000000
8,False,-0.063678,-3.028661,0.0,0.000000,0.000000
...,...,...,...,...,...,...
201151,False,-0.007615,-0.301597,-2.0,-0.305916,0.148683
201152,True,0.024124,-14.696128,-1.0,0.000000,0.000000
201153,True,-0.016562,1.251594,4.0,0.305248,-0.148683
201154,False,-0.007015,-1.604443,-5.0,-0.510826,0.000000


In [140]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
import random
seed = int(datetime.now().timestamp())
X = test_matches.iloc[:, 1:]
y = test_matches['radiant_win'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=seed)
log_reg_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])
naive_bayes_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GaussianNB())
])
support_vector_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC())
])
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)

In [141]:
cv_scores = cross_val_score(log_reg_pipeline, X_train, y_train, cv=cv, scoring='accuracy')
oof_predictions = cross_val_predict(log_reg_pipeline, X_train, y_train, cv=cv)
oof_proba       = cross_val_predict(log_reg_pipeline, X_train, y_train, cv=cv, method='predict_proba')
print(f"\nOOF Accuracy: {accuracy_score(y_train, oof_predictions):.3f}")
print(f"\nClassification Report (OOF):")
print(classification_report(y_train, oof_predictions, target_names=['Dire Win', 'Radiant Win']))


OOF Accuracy: 0.616

Classification Report (OOF):
              precision    recall  f1-score   support

    Dire Win       0.61      0.58      0.60     65492
 Radiant Win       0.62      0.65      0.63     68412

    accuracy                           0.62    133904
   macro avg       0.62      0.62      0.62    133904
weighted avg       0.62      0.62      0.62    133904



In [ ]:
cv_scores = cross_val_score(naive_bayes_pipeline, X_train, y_train, cv=cv, scoring='accuracy')
oof_predictions = cross_val_predict(naive_bayes_pipeline, X_train, y_train, cv=cv)
oof_proba       = cross_val_predict(naive_bayes_pipeline, X_train, y_train, cv=cv, method='predict_proba')
print(f"\nOOF Accuracy: {accuracy_score(y_train, oof_predictions):.3f}")
print(f"\nClassification Report (OOF):")
print(classification_report(y_train, oof_predictions, target_names=['Dire Win', 'Radiant Win']))


OOF Accuracy: 0.613

Classification Report (OOF):
              precision    recall  f1-score   support

    Dire Win       0.60      0.61      0.61     65492
 Radiant Win       0.62      0.61      0.62     68412

    accuracy                           0.61    133904
   macro avg       0.61      0.61      0.61    133904
weighted avg       0.61      0.61      0.61    133904



In [ ]:
cv_scores = cross_val_score(support_vector_pipeline, X_train, y_train, cv=cv, scoring='accuracy')
oof_predictions = cross_val_predict(support_vector_pipeline, X_train, y_train, cv=cv)
oof_proba       = cross_val_predict(support_vector_pipeline, X_train, y_train, cv=cv, method='predict_proba')
print(f"\nOOF Accuracy: {accuracy_score(y_train, oof_predictions):.3f}")
print(f"\nClassification Report (OOF):")
print(classification_report(y_train, oof_predictions, target_names=['Dire Win', 'Radiant Win']))

In [143]:
log_reg_pipeline.fit(X_train, y_train)
coefficients = pd.Series(
    log_reg_pipeline.named_steps['model'].coef_[0],
    test_matches.iloc[:, 1:].columns
)
print(coefficients)

draft_diff         0.429668
rating_diff        0.436621
form_diff          0.074858
roster_exp_diff   -0.007049
roster_age_diff   -0.029531
dtype: float64
